# PRCP-1002: Handwritten Digits Recognition

## Objective
Classify grayscale handwritten digit images (28 × 28) into ten classes (0–9) using MNIST.

## Project tasks
1. Perform exploratory data analysis (EDA) and explain the findings.
2. Prepare image data for classical machine-learning and deep-learning models.
3. Train and compare KNN, RBF-SVM, MLP (ANN), and CNN under a **fair primary experiment**.
4. Select the model using validation data, then evaluate the selected model once on the official test set.
5. Discuss limitations, computational trade-offs, and possible improvements.

**Fair-comparison rule:** All four models in the primary comparison use the same stratified 15,000-image training subset and the same 6,000-image validation set. A separate full-training-set experiment can be added later, but it must not be mixed into this primary ranking.

In [ ]:
# Reproducibility and imports
import os, sys, time, platform, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import tensorflow as tf

from tensorflow.keras.datasets import mnist
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("NumPy:", np.__version__, "| Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__, "| TensorFlow:", tf.__version__)
print("Random seed:", SEED)

## Task 1: Exploratory Data Analysis (EDA)

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print("Training images:", x_train.shape, "| labels:", y_train.shape)
print("Test images:    ", x_test.shape, "| labels:", y_test.shape)
print("Data type:", x_train.dtype)
print("Image dimensions:", f"{x_train.shape[1]} × {x_train.shape[2]}")
print("Pixel range:", int(x_train.min()), "to", int(x_train.max()))
print("Labels:", np.unique(y_train))
print("Missing values:", int(np.isnan(x_train.astype(np.float32)).sum()))
print("Mean / standard deviation:", round(float(x_train.mean()), 2), "/", round(float(x_train.std()), 2))
print("Zero-valued background pixels:", f"{(x_train == 0).mean():.2%}")

**Interpretation:** MNIST contains 60,000 training images and 10,000 official test images. Each image is a 28 × 28 grayscale array stored as `uint8`; pixel intensities range from 0 (black) to 255 (white). There are ten target classes. The zero-pixel proportion quantifies how much of each image is background; it should not be interpreted as a missing-data rate.

In [ ]:
# Inspect examples across all ten classes rather than only the first few records.
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit, ax in enumerate(axes.ravel()):
    idx = np.where(y_train == digit)[0][0]
    ax.imshow(x_train[idx], cmap="gray")
    ax.set_title(f"Digit {digit}")
    ax.axis("off")
plt.suptitle("One example from each digit class")
plt.tight_layout()
plt.show()

**Interpretation:** This grid confirms that the task contains ten visually distinct labels, while also showing that handwriting style varies within each label. A single example is illustrative, not representative of the full variation in a class.

In [ ]:
# Quantify class balance and show both counts and percentages.
class_counts = pd.Series(y_train).value_counts().sort_index()
class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage": 100 * class_counts / len(y_train)
})
display(class_summary.round(2))
print("Largest class:", int(class_counts.idxmax()), "with", int(class_counts.max()), "images")
print("Smallest class:", int(class_counts.idxmin()), "with", int(class_counts.min()), "images")
print("Count range:", int(class_counts.max() - class_counts.min()), "images")

ax = class_counts.plot(kind="bar", figsize=(8, 4))
ax.set(title="Training images per digit", xlabel="Digit", ylabel="Number of images")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

**Interpretation:** Use the displayed counts to identify the largest and smallest classes; do not infer this from the bar chart alone. Compare the count range and percentages to decide whether class imbalance is substantial. The class distribution should be considered when interpreting accuracy and macro-averaged metrics.

In [ ]:
# Mean image for each class: brighter pixels indicate locations commonly occupied by strokes.
fig, axes = plt.subplots(1, 10, figsize=(14, 2.5))
for digit, ax in enumerate(axes):
    mean_img = x_train[y_train == digit].mean(axis=0)
    ax.imshow(mean_img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(str(digit)); ax.axis("off")
plt.suptitle("Mean pixel-intensity image by digit")
plt.tight_layout(); plt.show()

**Interpretation:** Mean images reveal the typical stroke location and shape for each digit, but averaging also blurs differences in individual writing styles. Classes with similar average shapes may be visually harder to distinguish. The plot is descriptive and does not by itself establish which pairs are most frequently confused; that is assessed with a validation confusion matrix later.

In [ ]:
# Pixel-intensity distribution and per-image intensity summaries.
sample_pixels = x_train[::10].ravel()  # deterministic 10% sample for plotting
plt.figure(figsize=(8, 4))
plt.hist(sample_pixels, bins=32, range=(0, 255), edgecolor="black")
plt.title("Distribution of pixel intensities (10% deterministic sample)")
plt.xlabel("Pixel intensity"); plt.ylabel("Frequency")
plt.tight_layout(); plt.show()

per_image_mean = x_train.mean(axis=(1, 2))
plt.figure(figsize=(8, 4))
plt.hist(per_image_mean, bins=30, edgecolor="black")
plt.title("Distribution of mean intensity per image")
plt.xlabel("Mean pixel intensity"); plt.ylabel("Number of images")
plt.tight_layout(); plt.show()
print("Median image mean intensity:", round(float(np.median(per_image_mean)), 2))

**Interpretation:** The pixel histogram is expected to contain many low-intensity/background pixels and fewer bright stroke pixels. The per-image mean histogram shows how overall brightness varies across images. These distributions support the decision to scale pixel values before model training.

In [ ]:
# Duplicate-image diagnostic: exact pixel-array duplicates in the training set.
# Hashing flattened image bytes is deterministic; this checks exact duplicates, not near-duplicates.
hashes = [hashlib.md5(img.tobytes()).hexdigest() for img in x_train]
hash_counts = pd.Series(hashes).value_counts()
duplicate_groups = hash_counts[hash_counts > 1]
print("Images belonging to repeated exact-image groups:",
      int(duplicate_groups.sum()) if len(duplicate_groups) else 0)
print("Number of repeated exact-image groups:", len(duplicate_groups))
print("Note: visually similar but non-identical images are not counted.")

**EDA summary:** The dataset has fixed-size grayscale inputs, ten labels, no missing pixel values, and a high proportion of background pixels. Class counts and plots above provide quantitative support for balance claims. Mean images and intensity distributions describe common visual patterns. Exact-duplicate checking does not detect near-duplicates or establish whether two images were written by the same person.

## Task 2: Preprocessing and experimental design

- Convert pixel intensities from 0–255 to floating-point values in [0, 1].
- Flatten images to 784 features for KNN, SVM, and MLP; retain image structure for CNN.
- Split the original training data into a fit pool and validation set using stratification.
- Draw one stratified 15,000-image subset from the fit pool. **Every model uses exactly this subset** in the primary comparison.
- Keep the official test set untouched until the validation-based model choice is complete.

**Scaling rationale:** Dividing by 255 places all pixel features on a common numeric range and supports stable neural-network optimization. The classical models additionally use `StandardScaler` inside a scikit-learn `Pipeline`, fitted only on the training subset. This makes scaling part of the reproducible model workflow and avoids fitting preprocessing on validation or test data. Standardization can change the relative influence of rare pixels, so it is treated as an explicit modelling choice rather than a universally optimal transformation.

In [ ]:
# Normalize and prepare both representations.
x_train_norm = x_train.astype("float32") / 255.0
x_test_norm = x_test.astype("float32") / 255.0
x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat = x_test_norm.reshape(len(x_test_norm), -1)
x_train_cnn = x_train_norm[..., np.newaxis]
x_test_cnn = x_test_norm[..., np.newaxis]

fit_idx, val_idx = train_test_split(
    np.arange(len(y_train)), test_size=0.10, random_state=SEED, stratify=y_train
)
# Use the same fixed fit subset for all four models.
sub_idx, _ = train_test_split(
    fit_idx, train_size=15000, random_state=SEED, stratify=y_train[fit_idx]
)
X_fit_flat, y_fit = x_train_flat[sub_idx], y_train[sub_idx]
X_val_flat, y_val = x_train_flat[val_idx], y_train[val_idx]
X_fit_cnn, X_val_cnn = x_train_cnn[sub_idx], x_train_cnn[val_idx]

print("Common model-training subset:", X_fit_flat.shape, y_fit.shape)
print("Common validation set:", X_val_flat.shape, y_val.shape)
print("CNN input:", X_fit_cnn.shape)
print("Normalized pixel range:", float(x_train_norm.min()), "to", float(x_train_norm.max()))
print("Training subset class counts:")
display(pd.Series(y_fit).value_counts().sort_index().to_frame("Count"))

**Experimental design note:** The validation set has 6,000 images (10% of the original training pool). The four models are trained on the same 15,000 fit images and assessed on the same validation images. This controls the major training-size and validation-set differences in the primary comparison. The official 10,000-image test set is not used for tuning or model selection.

## Task 3: Model training and validation-based tuning

A small, documented hyperparameter comparison is used rather than claiming exhaustive optimization. Candidate settings are compared on the same validation set. Training and validation time are recorded. The selected configuration for each model family is the one with the highest validation macro-F1; accuracy is also reported. Macro-F1 gives each digit class equal weight.

In [ ]:
results = []
trained_models = {}
validation_predictions = {}
training_records = []

def score_predictions(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

def fit_and_score(name, model, X_fit, y_fit, X_val, y_val, config):
    start = time.perf_counter()
    model.fit(X_fit, y_fit)
    train_seconds = time.perf_counter() - start
    start = time.perf_counter()
    pred = model.predict(X_val)
    predict_seconds = time.perf_counter() - start
    row = {"Model": name, "Configuration": config,
           "Training samples": len(y_fit), "Training time (s)": train_seconds,
           "Validation prediction time (s)": predict_seconds}
    row.update(score_predictions(y_val, pred))
    training_records.append(row)
    return model, pred

### KNN: compare neighborhood sizes

In [ ]:
knn_candidates = []
for k in [3, 5, 7]:
    pipe = Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=k, weights="distance"))
    ])
    fitted, pred = fit_and_score("KNN", pipe, X_fit_flat, y_fit, X_val_flat, y_val,
                                 f"n_neighbors={k}, weights=distance")
    knn_candidates.append((fitted, pred))

### RBF Support Vector Machine: compare C values

In [ ]:
svm_candidates = []
for c_value in [2, 5, 10]:
    pipe = Pipeline([
        ("scale", StandardScaler()),
        ("model", SVC(kernel="rbf", C=c_value, gamma="scale"))
    ])
    fitted, pred = fit_and_score("SVM", pipe, X_fit_flat, y_fit, X_val_flat, y_val,
                                 f"RBF, C={c_value}, gamma=scale")
    svm_candidates.append((fitted, pred))

### MLP / ANN: compare compact architectures

In [ ]:
mlp_candidates = []
for hidden in [(128,), (128, 64)]:
    pipe = Pipeline([
        ("scale", StandardScaler()),
        ("model", MLPClassifier(hidden_layer_sizes=hidden, activation="relu",
                                max_iter=35, early_stopping=True,
                                validation_fraction=0.1, random_state=SEED))
    ])
    fitted, pred = fit_and_score("MLP", pipe, X_fit_flat, y_fit, X_val_flat, y_val,
                                 f"hidden_layers={hidden}, max_iter=35, early_stopping=True")
    mlp_candidates.append((fitted, pred))

### CNN: train on the same 15,000 images

In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
cnn_model = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(10, activation="softmax")
])
cnn_model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
cnn_model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)
start = time.perf_counter()
cnn_history = cnn_model.fit(
    X_fit_cnn, y_fit, epochs=10, batch_size=128,
    validation_data=(X_val_cnn, y_val),
    callbacks=[early_stop], verbose=1
)
cnn_train_seconds = time.perf_counter() - start
start = time.perf_counter()
cnn_val_prob = cnn_model.predict(X_val_cnn, verbose=0)
cnn_val_pred = np.argmax(cnn_val_prob, axis=1)
cnn_val_predict_seconds = time.perf_counter() - start
cnn_row = {"Model": "CNN", "Configuration": "2 Conv blocks; Dense(128); dropout=0.30; Adam; early stopping",
           "Training samples": len(y_fit), "Training time (s)": cnn_train_seconds,
           "Validation prediction time (s)": cnn_val_predict_seconds}
cnn_row.update(score_predictions(y_val, cnn_val_pred))
training_records.append(cnn_row)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(cnn_history.history["accuracy"], label="Training")
axes[0].plot(cnn_history.history["val_accuracy"], label="Validation")
axes[0].set_title("CNN accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(cnn_history.history["loss"], label="Training")
axes[1].plot(cnn_history.history["val_loss"], label="Validation")
axes[1].set_title("CNN loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

**CNN learning-curve interpretation:** Compare training and validation curves. A widening gap (training improves while validation stalls or worsens) can indicate overfitting. Early stopping monitors validation loss and restores the best observed weights. The CNN is trained on the same number of images as the classical models in this primary comparison.

In [ ]:
# Select the best configuration within each model family using validation macro-F1.
def best_candidate(candidates):
    scored = []
    for model, pred in candidates:
        scored.append((f1_score(y_val, pred, average="macro"), model, pred))
    return max(scored, key=lambda item: item[0])

for family, candidates in [
    ("KNN", knn_candidates), ("SVM", svm_candidates), ("MLP", mlp_candidates)
]:
    _, best_model, best_pred = best_candidate(candidates)
    trained_models[family] = best_model
    validation_predictions[family] = best_pred

trained_models["CNN"] = cnn_model
validation_predictions["CNN"] = cnn_val_pred

# Build one comparable row per selected model configuration.
all_records = pd.DataFrame(training_records)
selected_rows = []
for family in ["KNN", "SVM", "MLP"]:
    family_rows = all_records[all_records["Model"] == family]
    selected_rows.append(family_rows.loc[family_rows["Macro F1"].idxmax()])
selected_rows.append(all_records[all_records["Model"] == "CNN"].iloc[0])
validation_results = pd.DataFrame(selected_rows).reset_index(drop=True)
display(validation_results.round(4))

**Comparison interpretation:** The table reports the best validation configuration found for each model family, not an exhaustive global optimum. Every selected model used the same 15,000 training examples and the same validation set. Compare macro-F1 and accuracy together; inspect training and prediction time before choosing a deployment candidate.

In [ ]:
# Visual comparison of validation metrics and training/prediction cost.
validation_results.set_index("Model")[["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1.02)
)
plt.title("Validation metrics — equal training sample size")
plt.ylabel("Score"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

validation_results.set_index("Model")[["Training time (s)", "Validation prediction time (s)"]].plot(
    kind="bar", figsize=(10, 4)
)
plt.title("Measured training and validation prediction time")
plt.ylabel("Seconds"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

**Timing interpretation:** Timings are machine-, library-, and load-dependent; they are not universal benchmarks. Prediction time here covers the complete validation batch, not single-image latency. For a deployment decision, measure latency per image (including preprocessing), memory footprint, model size, throughput under realistic load, and hardware requirements.

## Validation diagnostics

In [ ]:
# Confusion matrix for the validation-selected model family.
selected_model_name = validation_results.sort_values("Macro F1", ascending=False).iloc[0]["Model"]
selected_val_pred = validation_predictions[selected_model_name]
print("Selected model by validation macro-F1:", selected_model_name)
cm_val = confusion_matrix(y_val, selected_val_pred, labels=np.arange(10))
plt.figure(figsize=(8, 7))
plt.imshow(cm_val, cmap="Blues")
plt.colorbar()
plt.xticks(range(10)); plt.yticks(range(10))
plt.xlabel("Predicted digit"); plt.ylabel("Actual digit")
plt.title(f"Validation confusion matrix — {selected_model_name}")
for r in range(10):
    for c in range(10):
        plt.text(c, r, str(cm_val[r, c]), ha="center", va="center", fontsize=7)
plt.tight_layout(); plt.show()

off_diag = cm_val.copy()
np.fill_diagonal(off_diag, 0)
top_pairs = np.dstack(np.unravel_index(np.argsort(off_diag.ravel())[::-1][:8], off_diag.shape))[0]
print("Most frequent validation confusions:")
for actual, predicted in top_pairs:
    if off_diag[actual, predicted] > 0:
        print(f"Actual {actual} predicted as {predicted}: {off_diag[actual, predicted]}")

**Diagnostic interpretation:** Off-diagonal cells show which digits the selected validation model confuses. The most frequent pairs are candidates for visual inspection and targeted error analysis. These are validation errors, so they can inform model development without consulting the official test set.

In [ ]:
# Display examples from the most frequent validation confusion pair.
if len(top_pairs):
    actual_digit, predicted_digit = top_pairs[0]
    matching = np.where((y_val == actual_digit) & (selected_val_pred == predicted_digit))[0][:10]
    if len(matching):
        fig, axes = plt.subplots(1, len(matching), figsize=(2 * len(matching), 2.5))
        if len(matching) == 1: axes = [axes]
        for ax, local_i in zip(axes, matching):
            original_i = val_idx[local_i]
            ax.imshow(x_train[original_i], cmap="gray")
            ax.set_title(f"True {actual_digit}\nPred {predicted_digit}")
            ax.axis("off")
        plt.suptitle("Examples from the most frequent validation confusion")
        plt.tight_layout(); plt.show()
    else:
        print("No errors found for the top confusion pair.")

## Final evaluation on the official test set

**Important:** Model family and hyperparameters have already been selected using validation data. The following cell evaluates only that selected model on the official test set. Do not use the test result to switch models or tune parameters; doing so would turn the test set into another validation set.

In [ ]:
# One-time official test evaluation of the validation-selected model.
if selected_model_name == "CNN":
    start = time.perf_counter()
    test_probabilities = cnn_model.predict(x_test_norm[..., np.newaxis], verbose=0)
    final_test_predictions = np.argmax(test_probabilities, axis=1)
    test_prediction_seconds = time.perf_counter() - start
else:
    start = time.perf_counter()
    final_test_predictions = trained_models[selected_model_name].predict(x_test_flat)
    test_prediction_seconds = time.perf_counter() - start

final_test_metrics = score_predictions(y_test, final_test_predictions)
print("Validation-selected model:", selected_model_name)
print("Official test metrics:")
display(pd.DataFrame([final_test_metrics]).round(4))
print("Official test prediction time (full 10,000-image batch):",
      round(test_prediction_seconds, 4), "seconds")
print("\nClassification report:")
print(classification_report(y_test, final_test_predictions, digits=4, zero_division=0))

### Test-set error analysis

In [ ]:
test_cm = confusion_matrix(y_test, final_test_predictions, labels=np.arange(10))
plt.figure(figsize=(8, 7))
plt.imshow(test_cm, cmap="Blues"); plt.colorbar()
plt.xticks(range(10)); plt.yticks(range(10))
plt.xlabel("Predicted digit"); plt.ylabel("Actual digit")
plt.title(f"Official test confusion matrix — {selected_model_name}")
for r in range(10):
    for c in range(10):
        plt.text(c, r, str(test_cm[r, c]), ha="center", va="center", fontsize=7)
plt.tight_layout(); plt.show()

wrong_test = np.where(final_test_predictions != y_test)[0]
print("Misclassified test images:", len(wrong_test), "of", len(y_test))
show_ids = wrong_test[:10]
if len(show_ids):
    fig, axes = plt.subplots(1, len(show_ids), figsize=(2 * len(show_ids), 2.5))
    if len(show_ids) == 1: axes = [axes]
    for ax, idx in zip(axes, show_ids):
        ax.imshow(x_test[idx], cmap="gray")
        ax.set_title(f"True {y_test[idx]}\nPred {final_test_predictions[idx]}")
        ax.axis("off")
    plt.suptitle("Examples misclassified by the selected model")
    plt.tight_layout(); plt.show()

**Test error interpretation:** Review the confusion matrix and sample errors for ambiguous handwriting, unusual stroke thickness, slant, or shapes that resemble another digit. This analysis describes performance on MNIST's held-out test split; it does not establish performance on handwriting collected from a different population or device.

## Model saving

In [ ]:
# Save the selected model in a format appropriate to its model family.
if selected_model_name == "CNN":
    cnn_model.save("handwritten_digits_cnn.keras")
    print("Saved: handwritten_digits_cnn.keras")
else:
    import joblib
    filename = f"handwritten_digits_{selected_model_name.lower()}.joblib"
    joblib.dump(trained_models[selected_model_name], filename)
    print("Saved:", filename)

# Model Comparison Report

## Experimental method
Four model families were considered: K-Nearest Neighbors (KNN), RBF Support Vector Machine (SVM), Multi-Layer Perceptron (MLP/ANN), and Convolutional Neural Network (CNN). For the primary comparison, all models were trained on the same stratified 15,000-image subset and evaluated on the same 6,000-image validation set. A small set of hyperparameters was compared for KNN, SVM, and MLP; the CNN used early stopping. The best configuration within each family was selected using validation macro-F1.

## Results
The validation results table and plots above are the authoritative comparison. The official test result below the validation analysis belongs only to the model selected using validation data. **Do not copy old accuracy figures into this report**; use the newly generated notebook outputs after running all cells.

## Recommendation framework
Choose a model based on validation macro-F1 and accuracy alongside measured inference time, model size, memory use, and deployment complexity. A CNN may be attractive for image structure learning, while an SVM or MLP may be preferable under different latency, memory, or operational constraints. The measured results in this run should determine the recommendation rather than an assumed ranking.

## Limitations
- The primary comparison uses a 15,000-image subset, not the full 60,000-image training set.
- Hyperparameter exploration is limited and is not exhaustive cross-validation.
- Timing depends on the runtime hardware, software versions, and system load.
- MNIST contains small, centered 28 × 28 grayscale images; real-world handwriting may differ in scale, orientation, background, writing instrument, and image quality.
- A high MNIST score does not guarantee performance on external handwriting data.
- Confidence scores, where available from a model, are not necessarily calibrated probabilities.

## Future improvements
Run controlled experiments with larger equal-size training subsets; perform cross-validation or a broader hyperparameter search; test augmentation (small rotations, shifts, and zoom); evaluate on a separate external handwriting dataset; calibrate confidence estimates; and benchmark model size and single-image latency on the intended deployment hardware.

# Challenges Faced and Techniques Used

| Challenge | Technique | Why it was used |
|---|---|---|
| Pixel values ranged from 0 to 255 | Converted pixels to floating point in [0, 1] | Provides a consistent input scale and supports neural-network optimization. |
| Classical estimators require vector inputs | Flattened each 28 × 28 image to 784 features | Matches the expected input shape of KNN, SVM, and MLP. |
| CNN requires spatial image structure | Kept image tensors in 28 × 28 × 1 format | Preserves local spatial relationships for convolution filters. |
| Feature scales can affect distance- and margin-based models | Used `StandardScaler` inside classical-model pipelines | Fits scaling only on training data and keeps preprocessing reproducible. |
| Unequal training sizes would make comparison unfair | Used the same stratified 15,000-image subset for all four models | Controls training sample count in the primary comparison. |
| Model tuning can overfit the test set | Used a validation split for configuration and model selection | Keeps the official test set for final evaluation only. |
| CNN may overfit | Used dropout, validation curves, and early stopping | Helps monitor generalization during training. |
| Similar digit shapes can be confused | Inspected confusion matrices and misclassified examples | Identifies difficult class pairs and qualitative failure modes. |
| Model quality is not only accuracy | Reported macro precision, recall, F1, and timing | Provides class-sensitive and computational context. |

# Conclusion

This project compares KNN, RBF-SVM, MLP, and CNN for MNIST handwritten digit recognition. The primary comparison uses the same 15,000 training images and the same validation set for all four model families, reducing the unequal-data limitation in the earlier experiment. Candidate hyperparameters are compared using validation macro-F1, and the selected model is evaluated once on the official test set.

The final recommendation should be based on the newly generated validation results and the practical trade-offs among predictive performance, inference speed, memory use, and deployment complexity. The official test result provides a final estimate for the selected model; it must not be used to revise the model choice.

The main limitation is that this experiment remains MNIST-specific and uses a modest training subset with limited hyperparameter exploration. Further work should include larger fair comparisons, broader tuning, data augmentation, external handwriting evaluation, and deployment benchmarking.